# Klasifikasi Jenis Sampah Menggunakan Convolutional Neural Network (CNN) dengan Backpropagation

**Program Python: Implementasi Manual dengan NumPy**

Disusun oleh:
1. Alfian Anggara Putra Afandy (181231043)
2. Bintang Cahya Putra Gunawan (181231061)

Program Studi S-1 Matematika, Departemen Matematika, Fakultas Sains dan Teknologi, Universitas Airlangga, 2026

---

Notebook ini berisi implementasi lengkap CNN dengan backpropagation untuk mengklasifikasikan citra sampah ke dalam dua kelas, yaitu **organik** (target $t = 0$) dan **anorganik** (target $t = 1$). Setiap langkah mengikuti Bab II dan Bab III makalah, mulai dari pengolahan citra, umpan maju, propagasi balik, sampai evaluasi model.

Seluruh perhitungan CNN ditulis sendiri menggunakan NumPy, tanpa TensorFlow, Keras, maupun PyTorch, sehingga setiap persamaan di makalah bisa ditelusuri langsung ke baris programnya. Bobot hasil pelatihan di akhir notebook diekspor ke website **PILAH** agar hasil prediksi di website sama dengan hasil di notebook ini.

Urutan isi notebook:
1. Import library
2. Parameter penelitian
3. Pengumpulan data (Langkah 2)
4. Pengolahan data citra (Langkah 3)
5. Fungsi-fungsi CNN (Langkah 4 dan 7)
6. Pelatihan CNN dengan backpropagation (Langkah 5)
7. Pengujian dan evaluasi model (Langkah 6)
8. Inferensi satu citra langkah demi langkah
9. Ekspor model ke website PILAH
10. Kesimpulan

## 1. Import Library

In [ ]:
import os
import glob
import json
import time
import random
import shutil
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from numpy.lib.stride_tricks import sliding_window_view

# Pengaturan tampilan angka dan grafik
np.set_printoptions(precision=4, suppress=True, linewidth=110)
plt.rcParams['figure.dpi'] = 110

print('Library berhasil diimport.')
print(f'Versi NumPy : {np.__version__}')

## 2. Parameter Penelitian

Seluruh nilai di bawah diambil dari Bab III makalah. Nilai `SEED` dikunci agar pengambilan data acak, pembagian data, dan inisialisasi bobot selalu sama setiap kali notebook dijalankan ulang.

In [ ]:
SEED = 42

# Data (Langkah 2)
JUMLAH_PER_KELAS = 1000      # 1.000 citra organik dan 1.000 citra anorganik
PROPORSI_LATIH   = 0.8       # 80% data latih, 20% data uji

# Pengolahan citra (Langkah 3)
UKURAN = 64                  # resize menjadi 64 x 64 piksel

# Arsitektur CNN (Langkah 4)
JUMLAH_KERNEL = 8
UKURAN_KERNEL = 3
STRIDE        = 1
PADDING       = (UKURAN_KERNEL - 1) // 2     # q = (k - 1) / 2 = 1, persamaan (2.9)
HIDDEN        = 16

# Parameter pelatihan
ALPHA     = 0.01             # learning rate
EPSILON   = 0.05             # batas error
E_MAKS    = 50               # maksimum epoch
THRESHOLD = 0.5              # y >= 0,5 -> anorganik, y < 0,5 -> organik

NAMA_KELAS = {0: 'Organik', 1: 'Anorganik'}

random.seed(SEED)
np.random.seed(SEED)

print('Parameter penelitian:')
print(f'  Data              : {JUMLAH_PER_KELAS} citra per kelas, latih {PROPORSI_LATIH:.0%} : uji {1 - PROPORSI_LATIH:.0%}')
print(f'  Ukuran input      : {UKURAN} x {UKURAN} (grayscale)')
print(f'  Konvolusi         : {JUMLAH_KERNEL} kernel {UKURAN_KERNEL} x {UKURAN_KERNEL}, stride {STRIDE}, padding {PADDING}')
print(f'  Max pooling       : 2 x 2, stride 2')
print(f'  Fully connected   : {HIDDEN} hidden (ReLU), 1 output (sigmoid)')
print(f'  alpha = {ALPHA}, epsilon = {EPSILON}, E_maks = {E_MAKS}')

## 3. Pengumpulan Data (Langkah 2)

Data diambil dari dataset **Waste Classification Data** (Sekar, 2019) di Kaggle:
https://www.kaggle.com/datasets/techsash/waste-classification-data

Dataset ini memuat 25.077 citra yang terbagi ke folder `O` (organik) dan `R` (recyclable, yang pada penelitian ini disebut anorganik). Label targetnya:
- `O` → organik, $t = 0$
- `R` → anorganik, $t = 1$

### a. Mengunduh Dataset

Dataset diunduh dengan `kagglehub`. Proses ini hanya perlu dilakukan sekali per sesi Colab (sekitar 1–2 menit).

Jika unduhan gagal, buka halaman dataset di Kaggle, unduh file ZIP-nya, unggah ke Colab, lalu ekstrak dan isi variabel `path_dataset` dengan lokasi folder hasil ekstrak.

In [ ]:
import kagglehub

path_dataset = kagglehub.dataset_download('techsash/waste-classification-data')
print('Dataset tersimpan di:', path_dataset)

In [ ]:
def kumpulkan_berkas(root):
    # Mencari seluruh citra di folder TRAIN/O, TRAIN/R, TEST/O, dan TEST/R.
    # Kunci (split, nama berkas) dipakai agar citra yang muncul di dua folder salinan tidak terhitung ganda.
    hasil = {'O': {}, 'R': {}}
    for split in ['TRAIN', 'TEST']:
        for kelas in ['O', 'R']:
            pola = os.path.join(root, '**', split, kelas, '*')
            for p in glob.glob(pola, recursive=True):
                if p.lower().endswith(('.jpg', '.jpeg', '.png')):
                    hasil[kelas][(split, os.path.basename(p))] = p
    return {kelas: sorted(isi.values()) for kelas, isi in hasil.items()}

berkas = kumpulkan_berkas(path_dataset)
total_asli = len(berkas['O']) + len(berkas['R'])

print('Komposisi dataset asli:')
print(f'  Organik   (O) : {len(berkas["O"])} citra')
print(f'  Anorganik (R) : {len(berkas["R"])} citra')
print(f'  Total         : {total_asli} citra')

### b. Pengambilan Acak 1.000 Citra per Kelas

Dari setiap kelas diambil 1.000 citra secara acak, sehingga diperoleh 2.000 citra dengan jumlah yang seimbang antara organik dan anorganik.

In [ ]:
pengacak = random.Random(SEED)
sampel = {
    0: pengacak.sample(berkas['O'], JUMLAH_PER_KELAS),   # organik, t = 0
    1: pengacak.sample(berkas['R'], JUMLAH_PER_KELAS),   # anorganik, t = 1
}

print(f'Citra organik terpilih   : {len(sampel[0])}')
print(f'Citra anorganik terpilih : {len(sampel[1])}')
print(f'Total                    : {len(sampel[0]) + len(sampel[1])}')

### c. Pembagian Data Latih dan Data Uji (80 : 20)

Pembagian dilakukan per kelas agar data latih dan data uji sama-sama seimbang, yaitu 800 latih dan 200 uji untuk setiap kelas.

In [ ]:
n_latih = int(JUMLAH_PER_KELAS * PROPORSI_LATIH)

data_latih, data_uji = [], []
for t in [0, 1]:
    data_latih += [(p, t) for p in sampel[t][:n_latih]]
    data_uji   += [(p, t) for p in sampel[t][n_latih:]]

pengacak.shuffle(data_latih)
pengacak.shuffle(data_uji)

def hitung(data, t):
    return sum(1 for _, label in data if label == t)

print(f'{"Kelas":<12}{"Target t":>9}{"Latih":>8}{"Uji":>7}{"Total":>8}')
print('-' * 44)
for t in [0, 1]:
    a, b = hitung(data_latih, t), hitung(data_uji, t)
    print(f'{NAMA_KELAS[t]:<12}{t:>9}{a:>8}{b:>7}{a + b:>8}')
print('-' * 44)
print(f'{"Total":<12}{"":>9}{len(data_latih):>8}{len(data_uji):>7}{len(data_latih) + len(data_uji):>8}')

In [ ]:
# Contoh citra dari masing-masing kelas
fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for baris, t in enumerate([0, 1]):
    contoh = [p for p, label in data_latih if label == t][:5]
    for kolom, p in enumerate(contoh):
        ax = axes[baris, kolom]
        ax.imshow(Image.open(p).convert('RGB'))
        ax.set_xticks([]); ax.set_yticks([])
        if kolom == 0:
            ax.set_ylabel(f'{NAMA_KELAS[t]} (t = {t})', fontsize=10)
fig.suptitle('Contoh Citra Data Latih', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Pengolahan Data Citra (Langkah 3)

Setiap citra diolah melalui tiga tahap agar ukurannya seragam, berkanal tunggal, dan bernilai kecil.

**a. Resize** menjadi $64 \times 64$ piksel.

**b. Grayscale**, persamaan (2.1):
$$\text{Gray} = 0{,}299R + 0{,}587G + 0{,}114B$$
Hasilnya dibulatkan ke bilangan bulat terdekat, seperti contoh di Subbab 2.3 ($159{,}25 \approx 159$).

**c. Normalisasi**, persamaan (2.2):
$$x_{\text{norm}} = \frac{x}{255}$$

Fungsi yang sama persis juga dijalankan oleh website PILAH di browser pengguna.

In [ ]:
def muat_citra(sumber):
    # Membuka citra, memutar sesuai orientasi kamera (EXIF), lalu mengubahnya ke RGB.
    # Citra transparan (PNG) ditempel di atas latar putih, sama seperti di website.
    img = Image.open(sumber)
    img = ImageOps.exif_transpose(img)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        img = img.convert('RGBA')
        latar = Image.new('RGBA', img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(latar, img)
    return img.convert('RGB')

def resize_citra(img, ukuran=UKURAN):
    # Langkah 3a: resize ke ukuran x ukuran piksel
    return img.resize((ukuran, ukuran), Image.BILINEAR)

def ke_grayscale(img_rgb):
    # Langkah 3b: persamaan (2.1), lalu dibulatkan ke bilangan bulat terdekat
    arr = np.asarray(img_rgb, dtype=np.float64)
    gray = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    return np.floor(gray + 0.5)

def normalisasi(gray):
    # Langkah 3c: persamaan (2.2)
    return gray / 255.0

def praproses(sumber):
    # Citra -> matriks input I berukuran 64 x 64 dengan elemen di [0, 1]
    return normalisasi(ke_grayscale(resize_citra(muat_citra(sumber))))

print('Fungsi pengolahan citra berhasil didefinisikan.')

In [ ]:
# Contoh satu piksel (Subbab 2.3)
R, G, B = 200, 150, 100
gray = 0.299 * R + 0.587 * G + 0.114 * B
gray_bulat = np.floor(gray + 0.5)

print(f'Piksel (R, G, B)  = ({R}, {G}, {B})')
print(f'Grayscale (2.1)   = 0,299({R}) + 0,587({G}) + 0,114({B}) = {gray:.2f}  ->  dibulatkan {gray_bulat:.0f}')
print(f'Normalisasi (2.2) = {gray_bulat:.0f} / 255 = {gray_bulat / 255:.4f}')

In [ ]:
# Tahapan pengolahan pada satu citra data latih
path_contoh, t_contoh = data_latih[0]
img_asli = muat_citra(path_contoh)
img_kecil = resize_citra(img_asli)
gray_kecil = ke_grayscale(img_kecil)
I_contoh = normalisasi(gray_kecil)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
axes[0].imshow(img_asli);                         axes[0].set_title(f'Citra asli\n{img_asli.size[0]} x {img_asli.size[1]}')
axes[1].imshow(img_kecil);                        axes[1].set_title('a. Resize\n64 x 64')
axes[2].imshow(gray_kecil, cmap='gray', vmin=0, vmax=255); axes[2].set_title('b. Grayscale\nnilai 0-255')
im = axes[3].imshow(I_contoh, cmap='viridis', vmin=0, vmax=1); axes[3].set_title('c. Normalisasi\nmatriks I, nilai 0-1')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes[3], fraction=0.046)
plt.tight_layout()
plt.show()

print(f'Kelas sebenarnya : {NAMA_KELAS[t_contoh]} (t = {t_contoh})')
print(f'Ukuran matriks I : {I_contoh.shape}')
print('Potongan 6 x 6 pertama matriks I:')
print(I_contoh[:6, :6])

Seluruh 2.000 citra kemudian diolah dengan fungsi yang sama. Hasilnya disimpan sebagai array berukuran $(N, 64, 64)$.

In [ ]:
def olah_semua(data):
    X = np.zeros((len(data), UKURAN, UKURAN))
    t = np.zeros(len(data), dtype=int)
    for i, (p, label) in enumerate(data):
        X[i] = praproses(p)
        t[i] = label
    return X, t

mulai = time.time()
X_latih, t_latih = olah_semua(data_latih)
X_uji, t_uji = olah_semua(data_uji)

print(f'Pengolahan selesai dalam {time.time() - mulai:.1f} detik.')
print(f'  X_latih : {X_latih.shape}, t_latih : {t_latih.shape}')
print(f'  X_uji   : {X_uji.shape}, t_uji   : {t_uji.shape}')
print(f'  Rentang nilai X_latih: {X_latih.min():.4f} sampai {X_latih.max():.4f}')

## 5. Fungsi-Fungsi CNN (Langkah 4 dan 7)

Arsitektur yang dibangun:

| Lapisan | Operasi | Ukuran keluaran | Banyak parameter |
|---|---|---|---|
| Input | matriks $I$ | $64 \times 64$ | – |
| Zero padding | $q = 1$ | $66 \times 66$ | – |
| Konvolusi + ReLU | 8 kernel $3 \times 3$, stride 1 | $8 \times 64 \times 64$ | $8 \cdot 9 + 8 = 80$ |
| Max pooling | $2 \times 2$, stride 2 | $8 \times 32 \times 32$ | – |
| Flatten | baris demi baris | $8.192$ | – |
| Hidden layer | 16 neuron, ReLU | $16$ | $8.192 \cdot 16 + 16 = 131.088$ |
| Output layer | 1 neuron, sigmoid | $1$ | $16 + 1 = 17$ |

Total parameter: **131.185**.

Notasi bobot mengikuti makalah: $K$ dan $b_c$ untuk kernel dan bias konvolusi, $v_{ij}$ dan $v_{0j}$ untuk bobot dan bias hidden layer, serta $w_j$ dan $w_0$ untuk bobot dan bias output layer.

### a. Fungsi Aktivasi dan Fungsi Error

- ReLU (2.5) dan turunannya (2.6)
- Sigmoid biner (2.7)
- Binary Cross-Entropy (2.17)

In [ ]:
def relu(x):
    # Persamaan (2.5): f(x) = max(0, x)
    return np.maximum(0, x)

def turunan_relu(x):
    # Persamaan (2.6): f'(x) = 1 jika x > 0, dan 0 jika x <= 0
    return (x > 0).astype(np.float64)

def sigmoid(x):
    # Persamaan (2.7). Ditulis dalam dua bentuk agar tidak overflow saat |x| besar.
    x = np.asarray(x, dtype=np.float64)
    return np.where(x >= 0, 1 / (1 + np.exp(-np.abs(x))), np.exp(-np.abs(x)) / (1 + np.exp(-np.abs(x))))

def bce(t, y):
    # Persamaan (2.17). y dibatasi agar ln(0) tidak terjadi.
    y = np.clip(y, 1e-7, 1 - 1e-7)
    return float(-(t * np.log(y) + (1 - t) * np.log(1 - y)))

print('Cek dengan contoh di makalah:')
print(f'  ReLU(-2), ReLU(0), ReLU(3) = {relu(np.array([-2, 0, 3]))}')
print(f'  sigmoid(0)  = {float(sigmoid(0)):.4f}')
print(f'  sigmoid(2)  = {float(sigmoid(2)):.4f}')
print(f'  sigmoid(-2) = {float(sigmoid(-2)):.4f}')

### b. Zero Padding, Konvolusi, dan Max Pooling

- Zero padding (2.9): menambahkan $q$ baris dan kolom bernilai 0 di setiap sisi.
- Konvolusi (2.10): $C_{i,j} = b_c + \sum_{u}\sum_{v} I_{i+u-1,\, j+v-1} K_{u,v}$. Program memakai `sliding_window_view` untuk mengambil semua jendela $3 \times 3$ sekaligus, lalu mengalikannya dengan kernel.
- Max pooling (2.12): mengambil nilai terbesar setiap jendela $2 \times 2$ dan mencatat posisinya untuk propagasi balik.

In [ ]:
def zero_padding(I, q=PADDING):
    # Persamaan (2.9)
    return np.pad(I, q, mode='constant', constant_values=0)

def konvolusi(I_pad, K, bc, s=STRIDE):
    # Persamaan (2.10) untuk seluruh kernel sekaligus.
    # I_pad : (H+2q, W+2q),  K : (jumlah_kernel, k, k),  bc : (jumlah_kernel,)
    k = K.shape[1]
    jendela = sliding_window_view(I_pad, (k, k))[::s, ::s]          # (H, W, k, k)
    C = np.einsum('ijuv,nuv->nij', jendela, K) + bc[:, None, None]  # (jumlah_kernel, H, W)
    return C, jendela

def max_pooling(R):
    # Persamaan (2.12), jendela 2 x 2 dengan stride 2.
    # posisi = indeks 0..3 nilai terbesar di tiap jendela (urutan: kiri-atas, kanan-atas, kiri-bawah, kanan-bawah)
    n, H, W = R.shape
    blok = R.reshape(n, H // 2, 2, W // 2, 2).transpose(0, 1, 3, 2, 4).reshape(n, H // 2, W // 2, 4)
    posisi = blok.argmax(axis=-1)
    P = np.take_along_axis(blok, posisi[..., None], axis=-1)[..., 0]
    return P, posisi

def kembalikan_pooling(dP, posisi):
    # Persamaan (2.24): error hanya diteruskan ke posisi nilai terbesar, posisi lain bernilai 0
    n, h, w = dP.shape
    blok = np.zeros((n, h, w, 4))
    np.put_along_axis(blok, posisi[..., None], dP[..., None], axis=-1)
    return blok.reshape(n, h, w, 2, 2).transpose(0, 1, 3, 2, 4).reshape(n, 2 * h, 2 * w)

print('Fungsi zero padding, konvolusi, dan max pooling berhasil didefinisikan.')

### c. Pengecekan dengan Contoh Hitung Manual

Sebelum dipakai pada data asli, fungsi-fungsi di atas diuji dengan angka contoh dari makalah dan PPT. Hasil program harus sama dengan hasil hitung manual.

In [ ]:
# Konvolusi: matriks 3 x 3 -> zero padding 5 x 5 -> konvolusi dengan kernel K dan bias 0,1
I_kecil = np.array([[0.62, 0.40, 0.15],
                    [0.55, 0.80, 0.30],
                    [0.20, 0.45, 0.90]])
K_kecil = np.array([[ 1, 0, -1],
                    [ 0, 1,  0],
                    [-1, 0,  1]], dtype=float)
I_kecil_pad = zero_padding(I_kecil)
C_kecil, _ = konvolusi(I_kecil_pad, K_kecil[None], np.array([0.1]))

print('Matriks hasil zero padding (5 x 5):')
print(I_kecil_pad)
print('\nJendela kiri atas dikalikan elemen kernel:')
print('C(1,1) = 0,1 + (0)(1) + (0)(0) + (0)(-1) + (0)(0) + (0,62)(1) + (0,40)(0) + (0)(-1) + (0,55)(0) + (0,80)(1)')
print(f'       = {C_kecil[0, 0, 0]:.2f}   (hitung manual di PPT: 1,52)')

# Max pooling: R 4 x 4 -> P 2 x 2
R_kecil = np.array([[1.52, 0.00, 0.21, 0.35],
                    [0.00, 0.48, 0.90, 0.00],
                    [0.12, 0.00, 0.00, 0.33],
                    [0.64, 0.27, 0.05, 0.00]])
P_kecil, _ = max_pooling(R_kecil[None])
print('\nHasil max pooling (2 x 2):')
print(P_kecil[0], '  (hitung manual di PPT: [[1,52 0,90] [0,64 0,33]])')

# Binary Cross-Entropy
print('\nBinary Cross-Entropy:')
print(f'  t = 1, y = 0,8  ->  L = {bce(1, 0.8):.4f}   (tebakan benar, error kecil)')
print(f'  t = 0, y = 0,8  ->  L = {bce(0, 0.8):.4f}   (tebakan salah, error besar)')

### d. Umpan Maju (Feedforward)

Fungsi `umpan_maju` menjalankan Langkah 5c (i) sampai (vii) untuk satu citra dan menyimpan seluruh hasil antaranya. Hasil antara ini dibutuhkan saat propagasi balik.

In [ ]:
def umpan_maju(I, param):
    I_pad = zero_padding(I)                                   # (i)   zero padding
    C, jendela = konvolusi(I_pad, param['K'], param['bc'])    # (ii)  konvolusi
    R = relu(C)                                               # (iii) ReLU
    P, posisi = max_pooling(R)                                # (iv)  max pooling + catat posisi
    x = P.reshape(-1)                                         # (v)   flatten: kernel ke-1 dulu, lalu baris demi baris
    z_in = param['v0'] + x @ param['V']                       # (vi)  persamaan (2.3)
    z = relu(z_in)                                            #       persamaan (2.4)
    y_in = param['w0'] + z @ param['w']                       # (vii) persamaan (2.14)
    y = float(sigmoid(y_in))                                  #       persamaan (2.15)
    return {'I_pad': I_pad, 'jendela': jendela, 'C': C, 'R': R, 'P': P, 'posisi': posisi,
            'x': x, 'z_in': z_in, 'z': z, 'y_in': float(y_in), 'y': y}

def prediksi_kelas(y):
    # Persamaan (2.16)
    return 1 if y >= THRESHOLD else 0

print('Fungsi umpan maju berhasil didefinisikan.')

### e. Propagasi Balik (Backpropagation)

Fungsi `propagasi_balik` menjalankan Langkah 5d dan menghasilkan koreksi $\Delta$ untuk setiap parameter:

| Langkah | Persamaan | Hasil |
|---|---|---|
| (i) Unit output | $\delta_k = t - y$ | $\Delta w_j = \alpha \delta_k z_j$, $\Delta w_0 = \alpha \delta_k$ |
| (ii) Hidden unit | $\delta_j = \delta_k w_j f'(z\_in_j)$ | $\Delta v_{ij} = \alpha \delta_j x_i$, $\Delta v_{0j} = \alpha \delta_j$ |
| (iii) Flatten | $\delta x_i = \sum_j \delta_j v_{ij}$ | disusun menjadi $\delta P$ |
| (iv) Max pooling | posisi maksimum saja | $\delta R$ |
| (v) ReLU | $\delta C = \delta R \cdot f'(C)$ | $\delta C$ |
| (vi) Konvolusi | $\Delta K_{u,v} = \alpha \sum_i \sum_j \delta C_{i,j} I_{i+u-1,\,j+v-1}$ | $\Delta K$, $\Delta b_c = \alpha \sum_i \sum_j \delta C_{i,j}$ |

Koreksi dihitung terlebih dahulu dengan bobot lama, lalu seluruh parameter diperbarui bersama pada Langkah 5e, $\theta_{\text{baru}} = \theta_{\text{lama}} + \Delta\theta$.

In [ ]:
def propagasi_balik(cache, t, param, alpha=ALPHA):
    # (i) Unit output, persamaan (2.18)-(2.19)
    delta_k = t - cache['y']
    dw = alpha * delta_k * cache['z']
    dw0 = alpha * delta_k

    # (ii) Hidden unit, persamaan (2.20)-(2.22)
    delta_in = delta_k * param['w']
    delta_j = delta_in * turunan_relu(cache['z_in'])
    dV = alpha * np.outer(cache['x'], delta_j)
    dv0 = alpha * delta_j

    # (iii) Error vektor flatten, persamaan (2.23), lalu disusun menjadi matriks dP
    delta_x = param['V'] @ delta_j
    dP = delta_x.reshape(cache['P'].shape)

    # (iv) Kembalikan ke posisi maksimum, persamaan (2.24)
    dR = kembalikan_pooling(dP, cache['posisi'])

    # (v) Turunan ReLU, persamaan (2.25)
    dC = dR * turunan_relu(cache['C'])

    # (vi) Koreksi kernel dan bias konvolusi, persamaan (2.26)-(2.27)
    dK = alpha * np.einsum('nij,ijuv->nuv', dC, cache['jendela'])
    dbc = alpha * dC.sum(axis=(1, 2))

    return {'K': dK, 'bc': dbc, 'V': dV, 'v0': dv0, 'w': dw, 'w0': dw0}

def perbarui_parameter(param, koreksi):
    # Langkah 5e, persamaan (2.28): parameter baru = parameter lama + koreksi
    for nama in param:
        param[nama] = param[nama] + koreksi[nama]
    return param

print('Fungsi propagasi balik dan pembaruan bobot berhasil didefinisikan.')

### f. Inisialisasi Parameter

Sesuai Langkah 5a, seluruh elemen kernel, bias, dan bobot diberi nilai acak pada rentang $[-0{,}5;\ 0{,}5]$.

In [ ]:
def inisialisasi_parameter(seed=SEED):
    rng = np.random.default_rng(seed)
    acak = lambda *ukuran: rng.uniform(-0.5, 0.5, size=ukuran)
    n_flatten = JUMLAH_KERNEL * (UKURAN // 2) * (UKURAN // 2)
    return {
        'K':  acak(JUMLAH_KERNEL, UKURAN_KERNEL, UKURAN_KERNEL),
        'bc': acak(JUMLAH_KERNEL),
        'V':  acak(n_flatten, HIDDEN),
        'v0': acak(HIDDEN),
        'w':  acak(HIDDEN),
        'w0': float(acak(1)[0]),
    }

param = inisialisasi_parameter()
jumlah = {nama: int(np.size(nilai)) for nama, nilai in param.items()}

print('Ukuran parameter:')
for nama, nilai in param.items():
    print(f'  {nama:<3}: bentuk {str(np.shape(nilai)):<12} -> {jumlah[nama]:>7} nilai')
print(f'\nLapisan konvolusi : {jumlah["K"] + jumlah["bc"]} parameter')
print(f'Hidden layer      : {jumlah["V"] + jumlah["v0"]} parameter')
print(f'Output layer      : {jumlah["w"] + jumlah["w0"]} parameter')
print(f'Total             : {sum(jumlah.values())} parameter')

### g. Pengecekan Gradien (Gradient Check)

Untuk memastikan rumus propagasi balik benar, koreksi dari program dibandingkan dengan turunan numerik. Karena $\delta$ merupakan negatif dari gradien, maka seharusnya

$$\frac{\Delta\theta}{\alpha} \approx -\frac{\partial L}{\partial \theta} \approx -\frac{L(\theta + h) - L(\theta - h)}{2h}$$

Jika selisih relatifnya sangat kecil (di bawah $10^{-4}$), rumus propagasi balik sudah benar. Selisih yang tidak persis nol berasal dari pembulatan bilangan komputer dan titik patah fungsi ReLU di $x = 0$.

In [ ]:
def cek_gradien(I, t, param, h=1e-7):
    cache = umpan_maju(I, param)
    koreksi = propagasi_balik(cache, t, param, alpha=1.0)   # alpha = 1 agar koreksi = -gradien
    rng = np.random.default_rng(0)
    print(f'{"Parameter":<14}{"Program":>14}{"Numerik":>14}{"Selisih relatif":>18}')
    print('-' * 60)
    selisih_maks = 0.0
    for nama in ['K', 'bc', 'V', 'v0', 'w', 'w0']:
        if nama == 'w0':
            daftar = [None]
        else:
            # pilih dua elemen yang gradiennya tidak nol agar perbandingan bermakna
            kandidat = np.argwhere(np.abs(koreksi[nama]) > 1e-9)
            pilih = rng.choice(len(kandidat), size=min(2, len(kandidat)), replace=False)
            daftar = [tuple(int(i) for i in kandidat[p]) for p in pilih]
        for indeks in daftar:
            nilai_awal = param['w0'] if indeks is None else param[nama][indeks]

            def L_dengan(nilai):
                if indeks is None:
                    param['w0'] = nilai
                else:
                    param[nama][indeks] = nilai
                return bce(t, umpan_maju(I, param)['y'])

            numerik = -(L_dengan(nilai_awal + h) - L_dengan(nilai_awal - h)) / (2 * h)
            L_dengan(nilai_awal)  # kembalikan nilai semula
            program = koreksi[nama] if indeks is None else koreksi[nama][indeks]
            relatif = abs(program - numerik) / max(1e-12, abs(program) + abs(numerik))
            selisih_maks = max(selisih_maks, relatif)
            label = nama if indeks is None else f'{nama}{list(indeks)}'
            print(f'{label:<14}{program:>14.6e}{numerik:>14.6e}{relatif:>18.2e}')
    return selisih_maks

# Dicek pada salah satu citra latih dengan parameter kecil agar sigmoid tidak jenuh
param_cek = inisialisasi_parameter(seed=7)
for nama in param_cek:
    param_cek[nama] = param_cek[nama] * 0.1
selisih = cek_gradien(X_latih[0], t_latih[0], param_cek)
print(f'\nSelisih relatif terbesar: {selisih:.2e}')
print('Propagasi balik BENAR.' if selisih < 1e-4 else 'Periksa kembali rumus propagasi balik.')

## 6. Pelatihan CNN dengan Backpropagation (Langkah 5)

Pelatihan memakai Stochastic Gradient Descent: parameter diperbarui setiap kali satu citra latih selesai diproses. Urutan citra latih diacak di awal setiap epoch (dengan `SEED` yang tetap) agar model tidak menghafal urutan data.

Pelatihan berhenti jika rata-rata error $\bar{L} < \varepsilon = 0{,}05$ atau epoch sudah mencapai $E_{\text{maks}} = 50$, persamaan (2.29).

Satu epoch di Colab membutuhkan sekitar 3–10 detik, sehingga seluruh pelatihan selesai dalam beberapa menit.

In [ ]:
param = inisialisasi_parameter()
pengacak_epoch = np.random.default_rng(SEED)
riwayat = {'epoch': [], 'L_rata': [], 'akurasi_latih': []}

print(f'Pelatihan dimulai: {len(X_latih)} citra latih, alpha = {ALPHA}, epsilon = {EPSILON}, E_maks = {E_MAKS}\n')
mulai_latih = time.time()

epoch = 1
while epoch <= E_MAKS:                                          # Langkah 5b
    mulai_epoch = time.time()
    urutan = pengacak_epoch.permutation(len(X_latih))
    total_L, benar = 0.0, 0

    for n in urutan:                                            # setiap data latih
        cache = umpan_maju(X_latih[n], param)                   # Langkah 5c
        total_L += bce(t_latih[n], cache['y'])                  # (viii) error BCE
        benar += int(prediksi_kelas(cache['y']) == t_latih[n])
        koreksi = propagasi_balik(cache, t_latih[n], param)     # Langkah 5d
        param = perbarui_parameter(param, koreksi)              # Langkah 5e

    L_rata = total_L / len(X_latih)                             # Langkah 5f, persamaan (2.29)
    akurasi_latih = benar / len(X_latih)
    riwayat['epoch'].append(epoch)
    riwayat['L_rata'].append(L_rata)
    riwayat['akurasi_latih'].append(akurasi_latih)
    print(f'Epoch {epoch:>2}/{E_MAKS} | L rata-rata = {L_rata:.4f} | akurasi latih = {akurasi_latih * 100:6.2f}% | {time.time() - mulai_epoch:.1f} detik')

    if L_rata < EPSILON:
        print(f'\nL rata-rata < epsilon ({EPSILON}), pelatihan dihentikan.')
        break
    epoch += 1

epoch_terakhir = riwayat['epoch'][-1]
waktu_latih = time.time() - mulai_latih
if epoch_terakhir == E_MAKS and riwayat['L_rata'][-1] >= EPSILON:
    print(f'\nEpoch sudah mencapai E_maks = {E_MAKS}, pelatihan dihentikan.')
print(f'Total waktu pelatihan: {waktu_latih / 60:.1f} menit')

# Simpan parameter optimal (Langkah 5g) agar tidak perlu melatih ulang
np.savez('parameter_optimal.npz', **param)
print('Parameter optimal disimpan ke parameter_optimal.npz')

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.2))
ax1.plot(riwayat['epoch'], riwayat['L_rata'], 'o-', color='#15803D', linewidth=2, markersize=4, label='L rata-rata')
ax1.axhline(EPSILON, color='#E5484D', linestyle='--', linewidth=1, label=f'epsilon = {EPSILON}')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Rata-rata error (BCE)')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(riwayat['epoch'], np.array(riwayat['akurasi_latih']) * 100, 's--', color='#D49A00', linewidth=1.5, markersize=3, label='Akurasi latih')
ax2.set_ylabel('Akurasi latih (%)')
ax2.set_ylim(0, 100)

garis = ax1.get_legend_handles_labels()
garis2 = ax2.get_legend_handles_labels()
ax1.legend(garis[0] + garis2[0], garis[1] + garis2[1], loc='center right')
ax1.set_title('Rata-rata Error dan Akurasi Latih per Epoch', fontsize=11, pad=10)
for sisi in ['top']:
    ax1.spines[sisi].set_visible(False); ax2.spines[sisi].set_visible(False)
plt.tight_layout()
plt.show()

print(f'L rata-rata epoch pertama : {riwayat["L_rata"][0]:.4f}')
print(f'L rata-rata epoch terakhir: {riwayat["L_rata"][-1]:.4f}')
print(f'Akurasi latih terakhir    : {riwayat["akurasi_latih"][-1] * 100:.2f}%')

Grafik di atas menunjukkan bagaimana error turun selama pelatihan. Jika garis hijau terus menurun, berarti backpropagation berhasil memperbaiki bobot dari epoch ke epoch. Naik-turun kecil masih wajar karena SGD memperbarui bobot setiap satu data, bukan setelah seluruh data.

## 7. Pengujian dan Evaluasi Model (Langkah 6)

Ke-400 citra uji diproses dengan umpan maju memakai parameter optimal. Kelas ditentukan dengan threshold 0,5, lalu disusun confusion matrix dengan **anorganik sebagai kelas positif**.

In [ ]:
mulai = time.time()
y_uji = np.array([umpan_maju(I, param)['y'] for I in X_uji])
t_prediksi = np.array([prediksi_kelas(y) for y in y_uji])
print(f'Pengujian {len(X_uji)} citra selesai dalam {time.time() - mulai:.1f} detik.')

def hitung_evaluasi(t_aktual, t_pred):
    TP = TN = FP = FN = 0
    for aktual, pred in zip(t_aktual, t_pred):
        if aktual == 1 and pred == 1:
            TP += 1   # anorganik diprediksi anorganik
        elif aktual == 0 and pred == 0:
            TN += 1   # organik diprediksi organik
        elif aktual == 0 and pred == 1:
            FP += 1   # organik keliru diprediksi anorganik
        elif aktual == 1 and pred == 0:
            FN += 1   # anorganik keliru diprediksi organik
    akurasi  = (TP + TN) / (TP + TN + FP + FN)                                           # (2.31)
    presisi  = TP / (TP + FP) if (TP + FP) > 0 else 0.0                                  # (2.32)
    recall   = TP / (TP + FN) if (TP + FN) > 0 else 0.0                                  # (2.33)
    f1       = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0.0  # (2.34)
    return TP, TN, FP, FN, akurasi, presisi, recall, f1

TP, TN, FP, FN, akurasi, presisi, recall, f1 = hitung_evaluasi(t_uji, t_prediksi)

print('=' * 64)
print('Evaluasi model CNN pada data uji')
print('=' * 64)
print('Komponen Confusion Matrix (anorganik = kelas positif):')
print(f'  True Positive  (TP) : {TP}')
print(f'  True Negative  (TN) : {TN}')
print(f'  False Positive (FP) : {FP}')
print(f'  False Negative (FN) : {FN}')
print('\nMetrik Evaluasi:')
print(f'  Accuracy  = ({TP} + {TN}) / ({TP} + {TN} + {FP} + {FN}) = {akurasi * 100:.2f}%')
print(f'  Precision = {TP} / ({TP} + {FP}) = {presisi * 100:.2f}%')
print(f'  Recall    = {TP} / ({TP} + {FN}) = {recall * 100:.2f}%')
print(f'  F1-score  = 2 x {presisi:.4f} x {recall:.4f} / ({presisi:.4f} + {recall:.4f}) = {f1 * 100:.2f}%')

In [ ]:
cm = np.array([[TP, FN],
               [FP, TN]])

fig, ax = plt.subplots(figsize=(5, 4.2))
ax.imshow(cm, cmap='Greens')
label = ['Anorganik (1)', 'Organik (0)']
ax.set_xticks([0, 1]); ax.set_xticklabels(label)
ax.set_yticks([0, 1]); ax.set_yticklabels(label)
nama_sel = [['TP', 'FN'], ['FP', 'TN']]
for i in range(2):
    for j in range(2):
        warna = 'white' if cm[i, j] > cm.max() / 2 else '#0B2B26'
        ax.text(j, i, f'{nama_sel[i][j]}\n{cm[i, j]}', ha='center', va='center', fontsize=13, color=warna)
ax.set_xlabel('Kelas prediksi', labelpad=6)
ax.set_ylabel('Kelas sebenarnya', labelpad=6)
ax.set_title('Confusion Matrix Data Uji', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Contoh hasil prediksi pada data uji (hijau = benar, merah = salah)
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, n in zip(axes.ravel(), range(12)):
    path_n, t_n = data_uji[n]
    ax.imshow(Image.open(path_n).convert('RGB'))
    benar = t_prediksi[n] == t_n
    ax.set_title(f'{NAMA_KELAS[t_prediksi[n]]}\ny = {y_uji[n]:.3f}', fontsize=9, color='#15803D' if benar else '#E5484D')
    for sisi in ax.spines.values():
        sisi.set_edgecolor('#15803D' if benar else '#E5484D'); sisi.set_linewidth(3)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Contoh Prediksi Data Uji (judul = kelas prediksi)', fontsize=12)
plt.tight_layout()
plt.show()

Cara membaca hasil evaluasi:
- **Accuracy** menunjukkan proporsi seluruh citra uji yang diklasifikasikan dengan benar.
- **Precision** menunjukkan seberapa sering prediksi "anorganik" memang benar anorganik.
- **Recall** menunjukkan seberapa banyak sampah anorganik yang berhasil dikenali.
- **F1-score** menyeimbangkan precision dan recall.

Model ini sengaja dibuat sederhana (satu lapisan konvolusi pada citra grayscale $64 \times 64$) supaya setiap langkahnya bisa dihitung manual. Karena itu, akurasinya wajar lebih rendah dibandingkan CNN berlapis banyak pada citra berwarna seperti di penelitian rujukan.

## 8. Inferensi Satu Citra Langkah demi Langkah

Bagian ini memperlihatkan apa yang terjadi pada satu citra dari awal sampai keluar kelasnya, memakai parameter optimal hasil pelatihan. Urutannya sama dengan proses di website PILAH ketika pengguna mengunggah foto.

Secara bawaan dipakai salah satu citra uji. Untuk mencoba foto sendiri, ubah `PAKAI_FOTO_SENDIRI` menjadi `True`, lalu jalankan sel di bawah dan pilih foto.

In [ ]:
PAKAI_FOTO_SENDIRI = False
INDEKS_CITRA_UJI = 0

if PAKAI_FOTO_SENDIRI:
    from google.colab import files
    unggahan = files.upload()
    sumber_inferensi = list(unggahan.keys())[0]
    t_inferensi = None
else:
    sumber_inferensi, t_inferensi = data_uji[INDEKS_CITRA_UJI]

print('Citra yang diproses :', os.path.basename(sumber_inferensi))
print('Kelas sebenarnya    :', NAMA_KELAS[t_inferensi] if t_inferensi is not None else 'tidak diketahui (foto sendiri)')

### Langkah 1: Citra input dan resize

In [ ]:
citra = muat_citra(sumber_inferensi)
citra_64 = resize_citra(citra)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
axes[0].imshow(citra);    axes[0].set_title(f'Citra asli ({citra.size[0]} x {citra.size[1]})')
axes[1].imshow(citra_64); axes[1].set_title('Setelah resize (64 x 64)')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### Langkah 2: Grayscale

In [ ]:
rgb_64 = np.asarray(citra_64)
gray_64 = ke_grayscale(citra_64)

r, g, b = rgb_64[0, 0]
print('Contoh piksel kiri atas:')
print(f'  (R, G, B) = ({r}, {g}, {b})')
print(f'  Gray = 0,299({r}) + 0,587({g}) + 0,114({b}) = {0.299 * r + 0.587 * g + 0.114 * b:.2f}  ->  {gray_64[0, 0]:.0f}')

plt.figure(figsize=(3.4, 3.4))
plt.imshow(gray_64, cmap='gray', vmin=0, vmax=255)
plt.title('Citra grayscale 64 x 64')
plt.axis('off')
plt.show()

### Langkah 3: Normalisasi menjadi matriks input I

In [ ]:
I = normalisasi(gray_64)
print(f'Ukuran matriks I : {I.shape}, rentang nilai {I.min():.4f} sampai {I.max():.4f}')
print(f'Contoh: {gray_64[0, 0]:.0f} / 255 = {I[0, 0]:.4f}')
print('\nPotongan 5 x 5 kiri atas matriks I:')
print(I[:5, :5])

### Langkah 4: Zero padding

In [ ]:
hasil = umpan_maju(I, param)   # seluruh hasil antara disimpan di sini dan ditampilkan satu per satu

print(f'Ukuran sebelum padding : {I.shape}')
print(f'Ukuran setelah padding : {hasil["I_pad"].shape}')
print('\nPotongan 5 x 5 kiri atas setelah padding (baris dan kolom pertama bernilai 0):')
print(hasil['I_pad'][:5, :5])

### Langkah 5: Konvolusi dengan 8 kernel

In [ ]:
K1, b1 = param['K'][0], param['bc'][0]
jendela_11 = hasil['I_pad'][0:3, 0:3]

print('Kernel ke-1 hasil pelatihan:')
print(K1)
print(f'Bias kernel ke-1: {b1:.4f}\n')
print('Jendela 3 x 3 kiri atas citra hasil padding:')
print(jendela_11)
manual = b1 + np.sum(jendela_11 * K1)
print(f'\nC(1,1) = b + jumlah(jendela x kernel) = {manual:.4f}')
print(f'C(1,1) dari fungsi konvolusi           = {hasil["C"][0, 0, 0]:.4f}')
print(f'Ukuran hasil konvolusi: {hasil["C"].shape} (8 feature map 64 x 64)')

batas = np.abs(hasil['C']).max()   # skala warna simetris agar 0 tepat di tengah (putih)
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(hasil['C'][k], cmap='RdBu_r', vmin=-batas, vmax=batas)
    ax.set_title(f'C kernel {k + 1}', fontsize=9); ax.axis('off')
plt.suptitle('Feature map hasil konvolusi (merah = positif, biru = negatif)', fontsize=10)
plt.tight_layout()
plt.show()

### Langkah 6: Aktivasi ReLU

In [ ]:
negatif = int((hasil['C'] < 0).sum())
print(f'Banyak elemen negatif yang diubah menjadi 0: {negatif} dari {hasil["C"].size} ({negatif / hasil["C"].size:.1%})')

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(hasil['R'][k], cmap='gray')
    ax.set_title(f'R kernel {k + 1}', fontsize=9); ax.axis('off')
plt.suptitle('Feature map setelah ReLU (terang = ciri yang kuat)', fontsize=10)
plt.tight_layout()
plt.show()

### Langkah 7: Max pooling 2 × 2

In [ ]:
print(f'Ukuran sebelum pooling : {hasil["R"].shape}')
print(f'Ukuran setelah pooling : {hasil["P"].shape}')
# Contoh satu jendela 2 x 2: diambil dari kernel dan posisi dengan nilai terbesar agar terlihat jelas
k, i, j = np.unravel_index(np.argmax(hasil['P']), hasil['P'].shape)
print(f'\nContoh jendela 2 x 2 kernel ke-{k + 1}, baris {2 * i + 1}-{2 * i + 2}, kolom {2 * j + 1}-{2 * j + 2}:')
print(hasil['R'][k, 2 * i:2 * i + 2, 2 * j:2 * j + 2], '-> nilai terbesar =', round(float(hasil['P'][k, i, j]), 4))

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(hasil['P'][k], cmap='gray')
    ax.set_title(f'P kernel {k + 1}', fontsize=9); ax.axis('off')
plt.suptitle('Hasil max pooling 32 x 32', fontsize=10)
plt.tight_layout()
plt.show()

### Langkah 8: Flatten

In [ ]:
x = hasil['x']
print(f'8 matriks 32 x 32 disusun menjadi satu vektor x dengan panjang {x.size}')
print(f'x1 sampai x8 : {x[:8]}')
print(f'Elemen x yang bernilai 0: {int((x == 0).sum())} dari {x.size}')

### Langkah 9: Hidden layer

In [ ]:
print('z_in_j = v0j + jumlah(x_i * v_ij), lalu z_j = max(0, z_in_j)\n')
print(f'{"j":>3}{"z_in":>12}{"z":>12}')
for j in range(HIDDEN):
    print(f'{j + 1:>3}{hasil["z_in"][j]:>12.4f}{hasil["z"][j]:>12.4f}')
print(f'\nHidden unit aktif (z > 0): {int((hasil["z"] > 0).sum())} dari {HIDDEN}')

plt.figure(figsize=(7, 2.6))
plt.bar(range(1, HIDDEN + 1), hasil['z'], color='#16A34A')
plt.xticks(range(1, HIDDEN + 1))
plt.xlabel('Hidden unit ke-j'); plt.ylabel('z_j')
plt.title('Output hidden layer')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Langkah 10: Output layer dan penentuan kelas

In [ ]:
y_in, y = hasil['y_in'], hasil['y']
kelas = prediksi_kelas(y)

print(f'y_in = w0 + jumlah(z_j * w_j) = {y_in:.4f}')
print(f'y    = 1 / (1 + e^(-y_in))   = {y:.4f}')
print(f'\nPeluang anorganik = {y * 100:.2f}%')
print(f'Peluang organik   = {(1 - y) * 100:.2f}%')
print(f'Karena y {">=" if y >= THRESHOLD else "<"} {THRESHOLD}, citra diklasifikasikan sebagai {NAMA_KELAS[kelas].upper()} (t = {kelas}).')
if t_inferensi is not None:
    print('Prediksi BENAR.' if kelas == t_inferensi else f'Prediksi SALAH, kelas sebenarnya {NAMA_KELAS[t_inferensi]}.')

sumbu = np.linspace(-8, 8, 300)
plt.figure(figsize=(6, 3.4))
plt.plot(sumbu, sigmoid(sumbu), color='#0B2B26', linewidth=2)
plt.axhline(THRESHOLD, color='#9CA3AF', linestyle='--', linewidth=1)
plt.scatter([np.clip(y_in, -8, 8)], [y], s=80, color='#F5B301' if kelas == 1 else '#16A34A', zorder=3)
plt.text(np.clip(y_in, -8, 8), y, f'  y = {y:.3f}  ', va='center', ha='right' if y_in > 4 else 'left')
plt.xlabel('y_in'); plt.ylabel('y = sigmoid(y_in)')
plt.title('Posisi citra pada kurva sigmoid')
plt.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Ekspor Model ke Website PILAH

Website PILAH hanya menjalankan umpan maju di browser pengguna, sehingga yang perlu dibawa hanyalah parameter optimal. Sel ini membuat:

- `model/model.bin`: 131.185 bilangan float32 dengan urutan $K$, $b_c$, $V$, $v_0$, $w$, $w_0$
- `model/meta.json`: ukuran arsitektur, threshold, metrik uji, dan riwayat pelatihan
- `model/golden.json`: beberapa citra uji beserta nilai $y$ dari notebook, untuk memastikan website menghasilkan $y$ yang sama
- `samples/`: contoh foto untuk tombol **Coba contoh** di website

Semua berkas dikemas menjadi `pilah_model_web.zip`. Isinya disalin ke folder `public/` proyek website.

In [ ]:
JUMLAH_CONTOH_PER_KELAS = 3
HANYA_CONTOH_YANG_BENAR = True   # True: contoh di website dipilih dari citra uji yang diprediksi benar

folder_ekspor = 'pilah_model_web'
shutil.rmtree(folder_ekspor, ignore_errors=True)
os.makedirs(f'{folder_ekspor}/model', exist_ok=True)
os.makedirs(f'{folder_ekspor}/samples', exist_ok=True)

# 1) model.bin: float32 little-endian dengan urutan K, bc, V, v0, w, w0
urutan_param = ['K', 'bc', 'V', 'v0', 'w', 'w0']
blob = np.concatenate([np.asarray(param[n], dtype='<f4').reshape(-1) for n in urutan_param]).tobytes()
assert len(blob) == 131_185 * 4, len(blob)
with open(f'{folder_ekspor}/model/model.bin', 'wb') as f:
    f.write(blob)

# Parameter yang benar-benar dipakai website (float32), untuk menghitung y pembanding
param_web = {n: np.asarray(param[n], dtype=np.float32).astype(np.float64) for n in urutan_param}
param_web['w0'] = float(param_web['w0'])

# 2) Contoh foto + golden test
contoh = []
for t in [1, 0]:
    kandidat = [n for n in range(len(data_uji)) if t_uji[n] == t and (t_prediksi[n] == t or not HANYA_CONTOH_YANG_BENAR)]
    contoh += kandidat[:JUMLAH_CONTOH_PER_KELAS]

manifest, golden = [], []
for nomor, n in enumerate(contoh, start=1):
    path_n, t_n = data_uji[n]
    ekstensi = os.path.splitext(path_n)[1].lower()
    nama_berkas = f'contoh-{nomor}{ekstensi}'
    shutil.copy(path_n, f'{folder_ekspor}/samples/{nama_berkas}')
    gray_n = ke_grayscale(resize_citra(muat_citra(path_n)))
    y_n = umpan_maju(gray_n / 255.0, param_web)['y']
    manifest.append({'src': f'/samples/{nama_berkas}', 'label': f'Contoh {NAMA_KELAS[t_n].lower()} {nomor}', 'kind': NAMA_KELAS[t_n].lower()})
    golden.append({'name': nama_berkas, 'file': f'/samples/{nama_berkas}', 't': int(t_n),
                   'gray': gray_n.astype(int).reshape(-1).tolist(), 'y': float(y_n)})

with open(f'{folder_ekspor}/samples/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
with open(f'{folder_ekspor}/model/golden.json', 'w') as f:
    json.dump(golden, f)

# 3) meta.json
meta = {
    'version': time.strftime('%Y.%m.%d'),
    'inputSize': UKURAN, 'kernels': JUMLAH_KERNEL, 'kernelSize': UKURAN_KERNEL, 'hidden': HIDDEN,
    'flatten': 'channel-major',
    'grayRound': True,
    'threshold': THRESHOLD,
    'uncertainBand': 0.1,
    'metrics': {'accuracy': akurasi, 'precision': presisi, 'recall': recall, 'f1': f1,
                'testSize': int(len(X_uji)), 'tp': TP, 'tn': TN, 'fp': FP, 'fn': FN},
    'training': {'trainSize': int(len(X_latih)), 'epochs': int(epoch_terakhir), 'alpha': ALPHA,
                 'epsilon': EPSILON, 'eMax': E_MAKS, 'seconds': round(waktu_latih, 1),
                 'lossHistory': [round(float(v), 5) for v in riwayat['L_rata']],
                 'accHistory': [round(float(v), 5) for v in riwayat['akurasi_latih']]},
    'trainedAt': time.strftime('%Y-%m-%d'),
}
with open(f'{folder_ekspor}/model/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Berkas ekspor:')
for root, _, nama_nama in os.walk(folder_ekspor):
    for nama in sorted(nama_nama):
        p = os.path.join(root, nama)
        print(f'  {p:<45} {os.path.getsize(p) / 1024:8.1f} KB')

In [ ]:
# Cek ulang: baca kembali model.bin lalu bandingkan y dengan parameter di memori
data_bin = np.frombuffer(open(f'{folder_ekspor}/model/model.bin', 'rb').read(), dtype='<f4').astype(np.float64)
o = 0
param_baca = {}
for n in urutan_param:
    ukuran = int(np.size(param[n]))
    param_baca[n] = data_bin[o:o + ukuran].reshape(np.shape(param[n]))
    o += ukuran
param_baca['w0'] = float(param_baca['w0'])

print(f'{"Contoh":<14}{"Kelas":<11}{"y notebook":>12}{"y model.bin":>13}')
for g in golden:
    y_baca = umpan_maju(np.array(g['gray'], dtype=np.float64).reshape(UKURAN, UKURAN) / 255.0, param_baca)['y']
    print(f'{g["name"]:<14}{NAMA_KELAS[g["t"]]:<11}{g["y"]:>12.6f}{y_baca:>13.6f}')
print('\nNilai y sama, model.bin siap dipakai website.')

In [ ]:
# Kemas lalu unduh
with zipfile.ZipFile('pilah_model_web.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, nama_nama in os.walk(folder_ekspor):
        for nama in nama_nama:
            p = os.path.join(root, nama)
            z.write(p, os.path.relpath(p, folder_ekspor))
print(f'pilah_model_web.zip dibuat ({os.path.getsize("pilah_model_web.zip") / 1024:.0f} KB).')

from google.colab import files
files.download('pilah_model_web.zip')

Langkah memasang model ke website:
1. Ekstrak `pilah_model_web.zip`.
2. Salin isi folder `model/` ke `pilah-web/public/model/` dan isi folder `samples/` ke `pilah-web/public/samples/`.
3. Jalankan `npm test` di folder proyek. Tes *golden* otomatis memeriksa bahwa website menghasilkan nilai $y$ yang sama dengan notebook ini.
4. Jalankan `npm run dev` untuk mencoba, lalu deploy ke Vercel.

## 10. Kesimpulan

In [ ]:
print('=' * 66)
print('RINGKASAN KLASIFIKASI JENIS SAMPAH DENGAN CNN DAN BACKPROPAGATION')
print('=' * 66)
print(f'Dataset          : Waste Classification Data (Sekar, 2019), {total_asli} citra')
print(f'Data dipakai     : {len(X_latih) + len(X_uji)} citra ({JUMLAH_PER_KELAS} per kelas)')
print(f'Pembagian data   : {len(X_latih)} latih, {len(X_uji)} uji')
print(f'Pengolahan citra : resize {UKURAN} x {UKURAN}, grayscale, normalisasi /255')
print(f'Arsitektur       : konvolusi {JUMLAH_KERNEL} kernel {UKURAN_KERNEL} x {UKURAN_KERNEL} + ReLU, max pooling 2 x 2,')
print(f'                   hidden {HIDDEN} neuron ReLU, output 1 neuron sigmoid')
print(f'Total parameter  : 131.185')
print(f'Pelatihan        : alpha = {ALPHA}, epsilon = {EPSILON}, berhenti di epoch {epoch_terakhir} dari {E_MAKS}')
print(f'L rata-rata akhir: {riwayat["L_rata"][-1]:.4f}')
print(f'Accuracy         : {akurasi * 100:.2f}%')
print(f'Precision        : {presisi * 100:.2f}%')
print(f'Recall           : {recall * 100:.2f}%')
print(f'F1-score         : {f1 * 100:.2f}%')
print('=' * 66)

benar_total = TP + TN
print('\nKesimpulan:')
print(f'1. CNN dengan backpropagation berhasil diterapkan untuk mengklasifikasikan citra sampah organik dan '
      f'anorganik. Pelatihan berjalan {epoch_terakhir} epoch dan rata-rata error turun dari '
      f'{riwayat["L_rata"][0]:.4f} menjadi {riwayat["L_rata"][-1]:.4f}.')
print(f'2. Seluruh langkah, mulai dari pengolahan citra, umpan maju, propagasi balik, sampai pembaruan bobot, '
      f'diimplementasikan secara manual dalam Python dengan NumPy. Pengecekan gradien menunjukkan rumus '
      f'propagasi balik sudah sesuai dengan turunan numeriknya.')
print(f'3. Pada {len(X_uji)} citra uji, model mengklasifikasikan {benar_total} citra dengan benar sehingga '
      f'accuracy = {akurasi * 100:.2f}%. Precision {presisi * 100:.2f}% berarti dari {TP + FP} citra yang diprediksi '
      f'anorganik, {TP} memang anorganik. Recall {recall * 100:.2f}% berarti dari {TP + FN} citra anorganik, '
      f'{TP} berhasil dikenali. F1-score yang menyeimbangkan keduanya sebesar {f1 * 100:.2f}%.')

Catatan untuk pembahasan:
- Model ini memakai satu lapisan konvolusi pada citra grayscale $64 \times 64$ agar setiap perhitungannya bisa ditelusuri secara manual. Informasi warna yang hilang saat grayscale dan ukuran citra yang kecil membatasi ciri yang bisa dipelajari, sehingga akurasinya wajar lebih rendah dari CNN berlapis banyak.
- Foto dari kamera ponsel bisa berbeda dari foto di dataset (latar, cahaya, sudut). Karena itu website PILAH menampilkan peluang setiap kelas dan memberi tanda "kurang yakin" jika $y$ dekat dengan 0,5.
- Bobot yang diekspor di bagian 9 sama persis dengan bobot yang menghasilkan metrik di atas, jadi hasil website dan hasil notebook dapat dipertanggungjawabkan bersama.